# SSCLNet Project Runner
Use this notebook to run the project in Google Colab.

## 1. Setup
First, make sure you have uploaded the `SSCLNet_Project.zip` file to the Files area on the left.
Then run this cell to unzip it.

In [ ]:
!unzip -q SSCLNet_Project.zip
%cd SSCLNet_Project

## 2. Install Dependencies

In [ ]:
!pip install -r requirements.txt

## 3. Train Models
Training is much faster in Colab if you enable GPU (Runtime > Change runtime type > T4 GPU).

In [ ]:
# Step 1: Self-Supervised Learning
!python train_ssl.py

In [ ]:
# Step 2: Brain MRI Classifier
!python train_brain_classifier.py

In [ ]:
# Step 3: Knee OA Classifier
!python train_knee_classifier.py

In [ ]:
# 1. Install Gradio
!pip install gradio

# 2. Create the App
import gradio as gr
import torch
import torch.nn.functional as F
from PIL import Image
from models.encoder import get_encoder
from models.classifier import get_brain_classifier, get_knee_classifier
from utils.augmentations import get_val_transform
import os

# Load Models
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Encoder
encoder = get_encoder()
if os.path.exists("models/encoder.pth"):
    encoder.load_state_dict(torch.load("models/encoder.pth", map_location=device, weights_only=True))

# Brain Model
brain_model = get_brain_classifier(get_encoder(), num_classes=4)
if os.path.exists("models/brain_model.pth"):
    brain_model.load_state_dict(torch.load("models/brain_model.pth", map_location=device, weights_only=True))
brain_model.to(device).eval()

# Knee Model
knee_model = get_knee_classifier(get_encoder(), num_classes=5)
if os.path.exists("models/knee_model.pth"):
    knee_model.load_state_dict(torch.load("models/knee_model.pth", map_location=device, weights_only=True))
knee_model.to(device).eval()

# Labels
BRAIN_CLASSES = ['Glioma', 'Meningioma', 'No Tumor', 'Pituitary']
KNEE_CLASSES = ['Grade 0', 'Grade 1', 'Grade 2', 'Grade 3', 'Grade 4']
transform = get_val_transform()

def predict(image, task):
    if image is None:
        return "Please upload an image."
    
    # Preprocess
    img_tensor = transform(Image.fromarray(image)).unsqueeze(0).to(device)
    
    with torch.no_grad():
        if task == "Brain MRI":
            output = brain_model(img_tensor)
            probs = F.softmax(output, dim=1)
            # Create dict for Gradio Label output
            return {BRAIN_CLASSES[i]: float(probs[0][i]) for i in range(len(BRAIN_CLASSES))}
        else:
            output = knee_model(img_tensor)
            probs = F.softmax(output, dim=1)
            return {KNEE_CLASSES[i]: float(probs[0][i]) for i in range(len(KNEE_CLASSES))}

# Create Interface
iface = gr.Interface(
    fn=predict,
    inputs=[
        gr.Image(type="numpy", label="Upload Medical Image"),
        gr.Radio(["Brain MRI", "Knee Osteoarthritis"], label="Select Task", value="Brain MRI")
    ],
    outputs=gr.Label(num_top_classes=3, label="Prediction"),
    title="SSCLNet Medical Image Analysis",
    description="Upload a Brain MRI or Knee X-ray to classify it."
)

# Launch with sharing enabled
iface.launch(share=True)